# Tutorial 01 — Random Interleaver

Turbo equalization exchanges extrinsic LLRs between an inner equalizer and an outer decoder. Both produce error events that are *bursty in their own domain* — adjacent in time at the equalizer, adjacent in the trellis at the decoder. A random interleaver between them makes those bursts look uncorrelated to the other component, which is the only reason the iteration converges.

This tutorial: build an interleaver, round-trip a vector through it, and visualise burst-error de-correlation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nsm.interleaver import random_indices, interleave, deinterleave
rng = np.random.default_rng(0)

## Build and round-trip

In [ ]:
N = 1024
idx = random_indices(N, seed=0)
x = rng.integers(0, 2, N).astype(np.int32)
y      = interleave(x, idx)
x_back = deinterleave(y, idx)
print('round-trip identity:', np.array_equal(x_back, x))

## Burst-error de-correlation

Create a vector with a contiguous run of errors (typical of a decoder mistake), interleave it, and look at where the errors end up. After interleaving they are spread uniformly — exactly what the next decoder needs to see.

In [ ]:
errors = np.zeros(N, dtype=int)
errors[100:130] = 1  # burst of 30 errors
errors_after = interleave(errors, idx)
fig, axes = plt.subplots(2, 1, figsize=(8, 3), sharex=True)
axes[0].stem(errors, basefmt=' '); axes[0].set_title('errors before interleaving (a burst)')
axes[1].stem(errors_after, basefmt=' '); axes[1].set_title('errors after interleaving')
for ax in axes: ax.set_yticks([])
plt.tight_layout()

Two random seeds give different permutations; the same seed reproduces.

In [ ]:
for seed in (1, 2, 1):
    print(seed, random_indices(8, seed=seed))